# MiniEngine on Google Colab

End-to-end runner for the **CS349D MiniEngine** repo. This notebook:

1. Verifies a GPU is attached
2. Clones the (public) GitHub repo
3. Installs dependencies
4. Loads your HuggingFace token from Colab secrets (for faster downloads)
5. Launches the OpenAI-compatible server in the background
6. Sends a sample chat request to verify it works
7. (Optional) Runs a quick serving / accuracy benchmark
8. Cleanly shuts the server down

## Before you start

**Enable GPU**: `Runtime → Change runtime type → Hardware accelerator → GPU` (T4 is free and is enough for `Qwen/Qwen3-4B-Instruct-2507` in bf16).

**Add your HuggingFace token to Colab secrets** (recommended — avoids the "unauthenticated requests" warning and rate limits):
1. Get a token at https://huggingface.co/settings/tokens (read access is enough).
2. In Colab, click the **🔑 key icon** in the left sidebar → **Add new secret**.
3. **Name:** `HF_TOKEN` &nbsp;&nbsp;**Value:** your token.
4. Toggle **Notebook access** on for this notebook.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone the repository

The repo is public, so no authentication is needed.

In [ ]:
import os
import shutil
import subprocess

GITHUB_USER = "bwathomas"  # @param {type:"string"}
REPO_NAME   = "MAST-miniengine-BWT"  # @param {type:"string"}
BRANCH      = "MAST-miniengine-BWT"  # @param {type:"string"}

WORKDIR = f"/content/{REPO_NAME}"

if os.path.isdir(WORKDIR):
    print(f"Removing existing checkout at {WORKDIR} \u2026")
    shutil.rmtree(WORKDIR)

clone_url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, WORKDIR],
    check=True,
)

%cd $WORKDIR
!git log --oneline -5

## 3. Install dependencies

Installs `miniengine` itself plus the benchmark extras. PyTorch is preinstalled on Colab so this is mostly small packages.

In [ ]:
!pip install -q -e .
!pip install -q -e ".[bench]" datasets requests

## 4. Load HuggingFace token from Colab secrets

Pulls the `HF_TOKEN` secret you set in the sidebar and exports it to the environment. This eliminates the "unauthenticated requests" warning and unlocks higher download rate limits.

If the secret isn't set, the cell warns and continues — anonymous downloads still work, just slower.

In [ ]:
def _load_hf_token() -> str | None:
    try:
        from google.colab import userdata
    except ImportError:
        return os.environ.get("HF_TOKEN")
    try:
        return userdata.get("HF_TOKEN")
    except Exception as e:
        print(f"  No HF_TOKEN secret available ({type(e).__name__}: {e}).")
        return None

hf_token = _load_hf_token()
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    masked = hf_token[:4] + "\u2026" + hf_token[-4:] if len(hf_token) > 8 else "set"
    print(f"  HF token loaded ({masked}).")
else:
    print("  Proceeding without an HF token \u2014 downloads may be rate-limited.")

### (Optional) Persist the HuggingFace model cache to Google Drive

Qwen3-4B weights are ~8 GB. Colab wipes `/root/.cache` between sessions, so subsequent runs would re-download. Mount Drive and point `HF_HOME` there to cache across sessions. **Skip this cell if you don't care about repeat-run speed.**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
print("HF_HOME =", os.environ["HF_HOME"])

## 5. Launch the server in the background

Starts `python -m miniengine` as a subprocess, redirecting all logs to `server.log`. Then polls `/health` until the server is responsive. **The model will download on first run (~8 GB), so the first wait can take 3–5 minutes.**

Change `MODEL` to use a different one. Suggestions:
- `Qwen/Qwen3-0.6B-Instruct-2507` — fastest, for smoke tests
- `Qwen/Qwen3-4B-Instruct-2507` — README default, good for T4
- `Qwen/Qwen3-8B` — milestone-2 target, needs L4 or better

In [ ]:
import signal
import subprocess
import time

import requests

MODEL   = "Qwen/Qwen3-4B-Instruct-2507"  # @param {type:"string"}
DTYPE   = "bfloat16"                      # @param ["bfloat16", "float16", "float32"]
MODE    = "batched"                       # @param ["baseline", "batched", "paged"]
PORT    = 8000
MAX_RUN = 16

log_path = os.path.join(WORKDIR, "server.log")
log_f = open(log_path, "w")

server_env = os.environ.copy()

server_proc = subprocess.Popen(
    [
        "python", "-u", "-m", "miniengine",
        "--model", MODEL,
        "--dtype", DTYPE,
        "--mode", MODE,
        "--host", "127.0.0.1",
        "--port", str(PORT),
        "--max-running", str(MAX_RUN),
    ],
    stdout=log_f,
    stderr=subprocess.STDOUT,
    start_new_session=True,
    cwd=WORKDIR,
    env=server_env,
)
print(f"Server PID = {server_proc.pid}.  Logs \u2192 {log_path}")

BASE_URL = f"http://127.0.0.1:{PORT}"
DEADLINE = time.time() + 600

while True:
    if server_proc.poll() is not None:
        print("\nServer crashed. Last 60 log lines:")
        print(subprocess.check_output(["tail", "-n", "60", log_path]).decode())
        raise RuntimeError("miniengine exited before becoming healthy")
    try:
        r = requests.get(f"{BASE_URL}/health", timeout=2)
        if r.status_code == 200:
            print("\nServer is healthy.")
            break
    except requests.exceptions.RequestException:
        pass
    if time.time() > DEADLINE:
        raise TimeoutError("Server did not become healthy within 10 minutes")
    print(".", end="", flush=True)
    time.sleep(5)

### Peek at the server log (handy while waiting / after a crash)

In [ ]:
!tail -n 40 server.log

## 6. Send a sample chat request

In [ ]:
import json

payload = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": "In one sentence, explain why batched decoding speeds up an LLM serving engine."}
    ],
    "max_tokens": 128,
    "temperature": 0.0,
    "stream": True,
}

print("\nStreaming response:\n")
with requests.post(f"{BASE_URL}/v1/chat/completions", json=payload, stream=True, timeout=120) as resp:
    resp.raise_for_status()
    for raw in resp.iter_lines():
        if not raw:
            continue
        line = raw.decode("utf-8")
        if not line.startswith("data: "):
            continue
        data = line[len("data: "):]
        if data == "[DONE]":
            print("\n\n[stream finished]")
            break
        chunk = json.loads(data)
        delta = chunk["choices"][0].get("delta", {})
        piece = delta.get("content", "")
        print(piece, end="", flush=True)

## 7. (Optional) Quick serving benchmark

Tiny configuration so it finishes in a few minutes. For a real measurement, increase `--num-requests` and add more concurrency levels.

In [ ]:
!python -m benchmark.bench_serving \
    --input-len 256 --output-len 128 \
    --concurrencies 1,4,8 \
    --num-requests 8 \
    --randomness 0.5

### Test — Milestone 2 Part A

Pre-allocated paged KV pool: print the pool's up-front capacity (pages + total KV tokens), then run `bench_serving` once with `--mode paged` at concurrency 8 and compare it against the M1 batched baseline at the same concurrency. Target: paging does not regress generation throughput vs. M1 batched. The cell stops the main launch-cell server (if running), spawns its own servers on `PORT`, then exits cleanly — re-run the launch cell after to restore your usual server.

In [ ]:
import os, sys, re, time, signal, subprocess, asyncio, io, contextlib

import numpy as np
import pandas as pd
import requests

TEST_PORT = PORT
TEST_BASE = f"http://127.0.0.1:{TEST_PORT}"
M2A_LOG_PAGED   = os.path.join(WORKDIR, "server_m2A_paged.log")
M2A_LOG_BATCHED = os.path.join(WORKDIR, "server_m2A_batched.log")

def _stop(p, lf=None):
    if p is None:
        return
    if p.poll() is None:
        try:
            os.killpg(os.getpgid(p.pid), signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            p.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(os.getpgid(p.pid), signal.SIGKILL)
            p.wait()
    if lf is not None:
        try:
            lf.close()
        except Exception:
            pass

# Free up the port + GPU memory used by the main launch cell, if any.
try:
    _stop(server_proc, locals().get("log_f"))
except NameError:
    pass
try:
    import torch
    torch.cuda.empty_cache()
except Exception:
    pass

def _launch(extra_args, log_path):
    lf = open(log_path, "w")
    p = subprocess.Popen(
        ["python", "-u", "-m", "miniengine",
         "--model", MODEL, "--dtype", DTYPE,
         "--host", "127.0.0.1", "--port", str(TEST_PORT),
         "--max-running", str(MAX_RUN)] + extra_args,
        stdout=lf, stderr=subprocess.STDOUT,
        start_new_session=True, cwd=WORKDIR,
    )
    deadline = time.time() + 900
    while True:
        if p.poll() is not None:
            tail = subprocess.check_output(["tail", "-n", "80", log_path]).decode()
            lf.close()
            raise RuntimeError(f"server died:\n{tail}")
        try:
            if requests.get(f"{TEST_BASE}/health", timeout=2).status_code == 200:
                break
        except requests.exceptions.RequestException:
            pass
        if time.time() > deadline:
            raise TimeoutError("server did not become healthy in 15 min")
        time.sleep(3)
    return p, lf

sys.path.insert(0, WORKDIR)
from benchmark import bench_serving as bs
from transformers import AutoTokenizer

def _bench(conc=8, num_requests=16, input_len=512, output_len=128):
    tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
    with contextlib.redirect_stdout(io.StringIO()):
        prompts = bs.load_wildchat_prompts(num_requests)
        reqs = bs.prepare_requests(
            prompts, tok, num_requests, input_len, output_len, randomness=0.5
        )
        results = asyncio.run(bs.run_at_concurrency(TEST_BASE, reqs, conc))
    ok = [r for r in results if r.error is None]
    ttfts   = np.array([r.ttft for r in ok if r.ttft is not None])
    compls  = np.array([r.completion_latency for r in ok if r.completion_latency is not None])
    total_out = sum(r.num_output_tokens for r in ok)
    total_time = max(r.end_time for r in ok) - min(r.start_time for r in ok)
    return {
        "gen_throughput_tok_s": (total_out / total_time) if total_time > 0 else float("nan"),
        "ttft_p50_ms":          float(np.percentile(ttfts, 50) * 1000) if len(ttfts)  else float("nan"),
        "completion_p50_ms":    float(np.percentile(compls, 50) * 1000) if len(compls) else float("nan"),
    }

paged_args = ["--mode", "paged", "--mem-fraction-static", "0.85", "--page-size", "32"]
p, lf = _launch(paged_args, M2A_LOG_PAGED)
try:
    time.sleep(1)
    log_text = open(M2A_LOG_PAGED).read()
    pool_lines = [l for l in log_text.splitlines() if "KV pool ready" in l]
    m = re.search(r"pages=(\d+).*page_size=(\d+).*total_kv_tokens=(\d+)", pool_lines[-1]) if pool_lines else None
    if m:
        print(
            f"Pre-allocated KV pool: {m.group(1)} pages × {m.group(2)} tokens/page "
            f"= {m.group(3)} total KV tokens"
        )
    else:
        print("Pre-allocated KV pool: (could not parse log)\n", log_text[-2000:])
    m2_paged = _bench(conc=8)
finally:
    _stop(p, lf)

p, lf = _launch(["--mode", "batched"], M2A_LOG_BATCHED)
try:
    m1_batched = _bench(conc=8)
finally:
    _stop(p, lf)

df = pd.DataFrame(
    [
        {"condition": "M1 batched baseline", **m1_batched},
        {"condition": "M2 paged",            **m2_paged},
    ]
).set_index("condition")
df

## 8. (Optional) Quick accuracy benchmark

In [ ]:
!python -m benchmark.bench_accuracy --dataset mmlu --num-samples 50 --concurrency 4

## 9. Stop the server

Always run this when you're done so the GPU memory is released and the runtime is reusable.

In [ ]:
if server_proc.poll() is None:
    try:
        os.killpg(os.getpgid(server_proc.pid), signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        server_proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        os.killpg(os.getpgid(server_proc.pid), signal.SIGKILL)
        server_proc.wait()
    print("Server stopped.")
else:
    print("Server already exited with code", server_proc.returncode)

log_f.close()